# 02 — Independent XGBoost results
Run notebook 00 first. Tune XGBoost across all four forecast setups using validation MAE, then refit XGBoost on training plus validation and evaluate the test sessions.
No Random Forest results or model comparison are required. Test scores never change the chosen hyperparameters.
Print and save validation/test MAE, RMSE and R², predictions and feature importance.


In [ ]:
%pip -q install numpy pandas scikit-learn joblib optuna xgboost


In [ ]:
from pathlib import Path
import sys
import importlib
from urllib.request import urlopen

helper_candidates = [Path.cwd() / "kc_common.py", Path.cwd() / "KC_CO2_Modular_Notebooks" / "kc_common.py"]
helper_path = next((p for p in helper_candidates if p.is_file()), None)
if helper_path is None:
    HELPER_REF = "main"
    helper_url = f"https://raw.githubusercontent.com/Namelessr679/ESP32-Indoor-Air-Quality-1.0Monitor/{HELPER_REF}/KC_CO2_Modular_Notebooks/kc_common.py"
    try:
        with urlopen(helper_url, timeout=30) as response:
            helper_source = response.read()
    except Exception as exc:
        raise RuntimeError("kc_common.py is unavailable. Place the matching helper beside the notebook or select a published HELPER_REF, then rerun setup.") from exc
    compile(helper_source, "kc_common.py", "exec")
    if b"def prepare_experiment(" not in helper_source:
        raise RuntimeError("The downloaded helper is incompatible. Use kc_common.py from the same repository revision as these notebooks.")
    helper_path = Path.cwd() / "kc_common.py"
    helper_path.write_bytes(helper_source)
sys.path.insert(0, str(helper_path.parent))
import kc_common
if Path(kc_common.__file__).resolve() != helper_path.resolve():
    raise RuntimeError("A different kc_common module is already loaded. Restart the runtime and rerun setup.")
importlib.reload(kc_common)
from kc_common import experiment_run_dir, load_prepared, run_tree_configurations, MODEL_FEATURES, make_preprocessor, make_tree_model, forecast_future_co2, result_row


In [ ]:
from google.colab import drive
import joblib
import pandas as pd

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
RUN_DIR = experiment_run_dir(PROJECT)


In [ ]:
prepared = load_prepared(RUN_DIR)


In [ ]:
def suggest_xgb_params(trial):
    params = {
        # Try different numbers of decision trees (100 to 500 in steps of 50)
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        # Try tree depths from 2 (very simple) up to 6 (more detailed/complex)
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        # The learning rate or step size shrinkage used to prevent overfitting (small steps are safer)
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        # Minimum sum of instance weight needed in a child node (helps prevent building overly specific rules)
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0, log=True),
        # Percentage of randomly chosen data rows to use when building each tree
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        # Percentage of randomly chosen features to use when building each tree
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        # Minimum loss reduction required to make a split (larger values prevent shallow splits)
        "gamma": trial.suggest_float("gamma", 0.0, 6.0),
        # L1 regularization term on weights (encourages weight values to go to zero)
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 3.0, log=True),
        # L2 regularization term on weights (smooths out weight changes)
        "reg_lambda": trial.suggest_float("reg_lambda", 0.2, 15.0, log=True),
    }
    return params


In [ ]:
xgb_results = run_tree_configurations("XGBoost", suggest_xgb_params, prepared, RUN_DIR / "xgb_validation.joblib")


## Final fitting, evaluation and export


In [ ]:
metric_rows, prediction_frames, importance_frames = [], [], []
for key, config in prepared["configurations"].items():
    horizon, approach = config["horizon"], config["approach"]
    validation_result = xgb_results[key]
    final_frame = pd.concat([config["train_df"], config["validation_df"]], ignore_index=True)
    if not set(final_frame.session_id).isdisjoint(set(config["test_df"].session_id)):
        raise ValueError("Test sessions overlap final fitting data.")
    final_preprocessor = make_preprocessor(scale_numeric=False)
    final_x = final_preprocessor.fit_transform(final_frame[MODEL_FEATURES])
    model = make_tree_model("XGBoost", validation_result["params"])
    model.fit(final_x, final_frame.target_change.to_numpy())
    test_prediction = forecast_future_co2(
        model, final_preprocessor, config["test_df"], horizon, approach,
        prepared["history_by_session"],
    )
    for split, frame, prediction in (
        ("Validation", config["validation_df"], validation_result["validation_predictions"]),
        ("Test", config["test_df"], test_prediction),
    ):
        metric_rows.append({"Horizon_min": horizon, "Approach": approach, "Split": split,
                            **result_row("XGBoost", frame, prediction)})
        exported = frame[["session_id", "time", "Actual"]].copy()
        exported["Predicted"] = prediction
        exported["Model"], exported["Split"] = "XGBoost", split
        exported["Horizon_min"], exported["Approach"] = horizon, approach
        prediction_frames.append(exported)
    importance = pd.DataFrame({"Feature": final_preprocessor.get_feature_names_out(),
                               "Importance": model.feature_importances_})
    importance["Model"] = "XGBoost"
    importance["Horizon_min"], importance["Approach"] = horizon, approach
    importance_frames.append(importance)

xgb_final_results = {
    "metadata": prepared["metadata"],
    "metrics": pd.DataFrame(metric_rows),
    "predictions": pd.concat(prediction_frames, ignore_index=True),
    "feature_importance": pd.concat(importance_frames, ignore_index=True),
}
if joblib.load(RUN_DIR / "preparation_metadata.joblib") != prepared["metadata"]:
    raise ValueError("Preparation changed during this run. Reload preparation and rerun notebook 02.")
joblib.dump(xgb_final_results, RUN_DIR / "xgb_results.joblib", compress=3)
xgb_final_results["metrics"].to_csv(RUN_DIR / "xgb_results.csv", index=False)
print(xgb_final_results["metrics"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Independent XGBoost results saved:", RUN_DIR / "xgb_results.joblib")
